# Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

df = pd.read_csv('../ml/df_model.csv', sep=';', dtype={'Code_INSEE': str})
display(df.head(5))

# Analyses univariées

In [ ]:
counts = df['Résultat'].value_counts()

plt.bar(range(len(counts)), counts.values, color=sns.color_palette("Set2", len(counts)), edgecolor='black', linewidth=0.5)
plt.title('Nombre de communes par bord politique')
plt.xlabel('Bord politique')
plt.ylabel('Nombre de communes')
plt.xticks(range(len(counts)), labels=counts.index)
plt.show()

La droite est beaucoup plus représentée que le centre et la gauche, il faudra tenir compte de ce déséquilibre pour le modèle

__Il faudra aussi hiérarchiser gauche-droite-centre dans le modèle, avec le centre au milieu, au lieu de faire un LabelEncoding sans hiérarchie__

# Matrice de corrélations

In [ ]:
target_dummies = pd.get_dummies(df['Résultat'])
df_corr = pd.concat([df.select_dtypes(include='number'), target_dummies], axis=1)
corr_matrix = df_corr.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(20, 15))

sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0, 
            square=True, mask=mask, cbar_kws={"shrink": .8}, 
            fmt='.2f', linewidths=0.5)

plt.title('Matrice de corrélations entre les données socio-économiques et le bord politique d\'une commune')
plt.tight_layout()
plt.show()

- Pas de corrélation vraiment signifiante
- Le taux de cadres et d'ouvriers pourrait avoir une certaine influence sur le vote
- Idem pour le taux de familles, couples avec enfant et personnes seules
- Idem pour les personnes mariées VS divorcés et célibataires
- Le genre et la tranche d'âge semblent n'avoir aucun impact sur le vote politique d'une commune

# Analyses bivariées

## Population

In [ ]:
sns.kdeplot(data=df, x='Population_active', hue='Résultat', 
            log_scale=True, fill=True, common_norm=False, 
            palette='icefire', alpha=0.5, linewidth=2)

plt.xlabel('Population active (log)', fontsize=12)
plt.ylabel('Densité de communes', fontsize=12)

plt.title('Population totale par bord politique')

ticks = [10, 100, 1000, 10000, 100000, 1000000]
plt.xticks(ticks, [f'{t:,}' for t in ticks])

plt.show()

- La droite est beaucoup plus présente dans les communes petites-moyennes (entre 200 et 1000 habitants)
- La gauche est plus présente dans les très grandes villes (à partir de 10000 habitants)
- Le centre est entre les deux, pas de caractéristique distinctive
- Au lieu de garder la population telle quelle, peut-être un booléen grande_ville (> 10000 habitants)

## Genre

In [ ]:
sns.boxplot(data=df, x=df['Résultat'], y=df['Femmes'], hue=df['Résultat'], palette='Set2', showfliers=True)

plt.title('Distribution de la proportion de femmes par bord politique')
plt.xlabel('Bord politique')
plt.ylabel('Proportion de femmes')

plt.show()

Le genre majoritaire dans une commune n'a aucun impact sur le bord politique

## Catégories socio-professionnelles

In [ ]:
df_csp = df[['Résultat', 'Population_active', 'Agriculteurs', 'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers', 'Etudiants', 'Retraités', 'Inactifs']]
df_csp['Total_actifs'] = df_csp['Agriculteurs'] + df_csp['Artisans'] + df_csp['Cadres'] + df_csp['Intermédiaires'] + df_csp['Employés'] + df_csp['Ouvriers']


sns.kdeplot(data=df_csp, x='Total_actifs', hue='Résultat', 
            log_scale=False, fill=True, common_norm=False, 
            palette='icefire', alpha=0.5, linewidth=2)

plt.xlabel('Proportion d\'actifs', fontsize=12)
plt.ylabel('Densité de communes', fontsize=12)

plt.title('Proportion d\'actifs par bord politique')

plt.show()

- Entre 50 et 70 % d'actifs dans une commune, la droite est plus présente
- Entre 25 et 50 % d'actifs dans une commune, la gauche est plus présente
- Le centre est entre les deux
- On pourrait créer une variable booléenne majorité_actifs (True si la proportion > 50 %, sinon False), ou simplement ajouter la proportion d'actifs

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 24), layout="constrained")
fig.suptitle('Comparaison des distributions de catégories socio-politiques par bord politique', fontsize=16, fontweight='bold')

comparisons = [
    ('Agriculteurs', 'Agriculteurs par bord politique'),
    ('Artisans', 'Artisans et commerçants par bord politique'),
    ('Cadres', 'Cadres par bord politique'),
    ('Intermédiaires', 'Professions intermédiaires par bord politique'),
    ('Employés', 'Employés par bord politique'),
    ('Ouvriers', 'Ouvriers par bord politique'),
    ('Retraités', 'Retraités par bord politique'),
    ('Inactifs', 'Autres inactifs par bord politique')
]

for i, (num_var, title) in enumerate(comparisons):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    
    sns.boxplot(data=df_csp, x='Résultat', y=num_var, ax=ax, 
                hue='Résultat', showfliers=True)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Résultat'.title())
    ax.set_ylabel(num_var.title())
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

# plt.tight_layout()
plt.show()

- Les communes ayant moins de cadres ou moins de retraités s'éloignent de la droite
- Les communes ayant plus d'ouvriers ou d'employés votent davantage à droite
- Les communes avec davantage d'inactifs votent plutôt à gauche

In [ ]:
cols_groups = ['Agriculteurs', 'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers', 'Etudiants', 'Retraités', 'Inactifs']
mask = df_csp[cols_groups].notna().any(axis=1)
df_csp.loc[mask, 'Majorité'] = df_csp.loc[mask, cols_groups].idxmax(axis=1)

# 1. Préparation des données : on calcule le % de chaque CSP au sein de chaque bord
df_counts = df_csp.groupby(['Résultat', 'Majorité']).size().reset_index(name='count')
df_counts['densité'] = df_counts.groupby('Résultat')['count'].transform(lambda x: x / x.sum() * 100)


plt.figure(figsize=(12, 8))

sns.barplot(
    data=df_counts, 
    y='Majorité',      # Les CSP sur l'axe Y
    x='densité',       # La proportion sur l'axe X
    hue='Résultat',    # Les 3 barres par catégorie
    palette='icefire'
)

plt.title('Profil sociologique des communes par bord politique', fontweight='bold', fontsize=14)
plt.xlabel('Proportion de communes (%)')
plt.ylabel('CSP majoritaire dans la commune')
plt.legend(title='Bord politique')
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.show()

- Les communes ayant une majorité de cadres, artisans ou agriculteurs s'éloignent de la droite
- Les communes avec majorité d'ouvriers, employés ou professions intermédiaires tendent vers la droite
- Les communes avec une majorité d'autres inactifs votent plutôt à gauche
- Les communes avec une majorité de retraités votent plutôt au centre

- Peut-être ajouter cette variable si le modèle ne trouve pas de features distinctives avec les catégories séparées.

## Tranches d'âge

In [ ]:
df_age = df[['Résultat', '15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +']]

fig, axes = plt.subplots(3, 2, figsize=(16, 20), layout="constrained")
fig.suptitle('Comparaison des tranches d\'âge par bord politique', fontsize=16, fontweight='bold')

comparisons = [
    ('15-24 ans', '15-24 ans par bord politique'),
    ('25-39 ans', '25-39 ans par bord politique'),
    ('40-54 ans', '40-54 ans par bord politique'),
    ('55-64 ans', '55-64 ans par bord politique'),
    ('65-79 ans', '65-79 ans par bord politique'),
    ('80 ans et +', '80 ans et + par bord politique')
]

for i, (var, title) in enumerate(comparisons):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    
    sns.boxplot(data=df_age, x='Résultat', y=var, ax=ax, 
                hue='Résultat', showfliers=True)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Résultat'.title())
    ax.set_ylabel(var.title())
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.show()

Pas d'observation vraiment différenciante. Quelques variations, mais très légères.

In [ ]:
cols_ages = ['15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +']
mask = df_age[cols_ages].notna().any(axis=1)
df_age.loc[mask, 'Majorité'] = df_age.loc[mask, cols_ages].idxmax(axis=1)

df_counts_ages = df_age.groupby(['Résultat', 'Majorité']).size().reset_index(name='count')
df_counts_ages['Densité'] = df_counts_ages.groupby('Résultat')['count'].transform(lambda x: x / x.sum() * 100)


plt.figure(figsize=(12, 8))

sns.barplot(
    data=df_counts_ages, 
    y='Majorité',      # Les CSP sur l'axe Y
    x='Densité',       # La proportion sur l'axe X
    hue='Résultat',    # Les 3 barres par catégorie
    palette='icefire'
)

plt.title('Tranches d\'âges des communes par bord politique', fontweight='bold', fontsize=14)
plt.xlabel('Proportion de communes (%)')
plt.ylabel('Tranche d\'âge majoritaire dans la commune')
plt.legend(title='Bord politique')
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.show()

- Les communes avec une majorité de 40-54 ans tendent vers la droite, tandis que celles avec une majorité de 65-79 ans s'en éloignent.
- Cette feature semble plus pertinente que les proportions pour chaque tranche d'âge distincte.

## Composition du ménage

In [ ]:
df_household = df[['Résultat', 'Personne seule', 'Homme seul', 'Femme seule', 'Colocation', 'Famille', 'Famille monoparentale', 'Couple sans enfant', 'Couple avec enfants']]

fig, axes = plt.subplots(3, 2, figsize=(16, 20), layout="constrained")
fig.suptitle('Comparaison des compositions de ménages par bord politique', fontsize=16, fontweight='bold')

comparisons = [
    ('Homme seul', 'Hommes seuls par bord politique'),
    ('Femme seule', 'Femmes seules par bord politique'),
    ('Colocation', 'Colocations par bord politique'),
    ('Famille monoparentale', 'Familles monoparentales par bord politique'),
    ('Couple sans enfant', 'Couples sans enfant par bord politique'),
    ('Couple avec enfants', 'Couples avec enfants par bord politique')
]

for i, (var, title) in enumerate(comparisons):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    
    sns.boxplot(data=df_household, x='Résultat', y=var, ax=ax, 
                hue='Résultat', showfliers=True)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Résultat'.title())
    ax.set_ylabel(var.title())
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
sns.kdeplot(data=df_household, x='Personne seule', hue='Résultat', 
            log_scale=False, fill=True, common_norm=False, 
            palette='icefire', alpha=0.5, linewidth=2)

plt.xlabel('Proportion de personnes seules', fontsize=12)
plt.ylabel('Densité de communes', fontsize=12)

plt.title('Personnes seules par bord politique')

plt.show()


- Les communes avec plus de personnes seules votent davantage à gauche.
- Les couples sans enfant et les familles monoparentales votent un peu plus à gauche.
- À l'inverse, les couples avec enfants s'éloignent de la gauche.

## Statut marital

In [ ]:
df_conjugality = df[['Résultat', 'Mariés', 'Pacsés', 'Concubinage', 'Veufs', 'Divorcés', 'Célibataires']]

fig, axes = plt.subplots(3, 2, figsize=(16, 20), layout="constrained")
fig.suptitle('Comparaison des statuts maritaux par bord politique', fontsize=16, fontweight='bold')

comparisons = [
    ('Mariés', 'Personnes mariées par bord politique'),
    ('Pacsés', 'Personnes pacsées par bord politique'),
    ('Concubinage', 'Personnes en concubinage par bord politique'),
    ('Veufs', 'Personnes veuves par bord politique'),
    ('Divorcés', 'Personnes divorcées par bord politique'),
    ('Célibataires', 'Personnes célibataires par bord politique')
]

for i, (var, title) in enumerate(comparisons):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    
    sns.boxplot(data=df_conjugality, x='Résultat', y=var, ax=ax, 
                hue='Résultat', showfliers=True)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Résultat'.title())
    ax.set_ylabel(var.title())
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.show()

- Les personnes célibataires et divorcées votent davantage pour la gauche
- Redondance avec les observations sur la composition des ménages (les personnes seules votent davantage à gauche)
- Les personnes mariées et pacsées votent moins pour la gauche
- Les personnes en concubinage votent moins pour le centre

# Recommandations pour le modèle

## Observations générales

- La plupart des analyses montrent une **hiérarchie entre les trois groupes** : plus on s'éloigne de la droite, plus on va vers la gauche en passant par le centre, et inversement. Il faudra donc tester un **ordinal encoding** pour la variable cible au lieu d'un label encoding qui traiterait les trois groupes comme totalement indépendants les uns des autres.

- Dans le même esprit, au lieu de prédire uniquement un résultat, il serait plus pertinent d'**afficher les pourcentages prédits pour chaque groupe**.

- La **variable cible est très déséquilibrée**, avec énormément de communes pour la droite, peu pour la gauche et très peu pour le centre. Il faudra compenser ce déséquilibre en utilisant un c**lass_weight balanced** ou avec **Smote**, selon le modèle utilisé.

- L'analyse *n'a pas mis en lumière des interactions entre variables* pouvant apporter de la valeur pour le modèle, toutefois il faudra le vérifier avec les **polynomial features**.


## Propositions de features

- **'grande ville'** > 10000 habitants (booléen)
- **'proportion d'actifs'** (float, somme des proportions d'actifs) ou **'majorité d'actifs'** (booléen)
- **'csp majoritaire'** (str à one-hot encoder ou label encoder)
- **'tranche d'âge majoritaire'** (str à one-hot encoder ou label encoder)
- *supprimer 'hommes seuls' et 'femmes seules'* pour garder uniquement 'personnes seules' et éviter la redondance
- exclure les colonnes inutiles avec la stratégie **select k best**